In [0]:
import logging
from pyspark.sql.functions import col
def configure_logging(level=logging.INFO):
    root_logger = logging.getLogger()
    if not root_logger.handlers:
        logging.basicConfig(
            level=level,
            format='%(asctime)s - %(levelname)s - %(name)s - %(message)s'
        )
    root_logger.setLevel(level)
    return root_logger

def get_logger(name):
    configure_logging()
    return logging.getLogger(name)

logger = get_logger(__name__)

default_key_cols="execution_id,model,layer,topic,father_topic"
instruction_activity_list=["model_evaluation", "topic_naming"]
refinement_queue_table_name="topics_to_refine"
refinement_min_topic_count=100
refinement_score_threshold=4
refinement_max_layer=1
checkpoint_path_prefix_template = "/Volumes/{catalog}/{schema}/checkpoints/{stream_table_name}"
cluster_model_list=["HDBSCAN", "KMEANS"]
write_mode_list=["append", "overwrite"]
execution_keys_list=["cluster_model", "timestamp"]
catalog_list=["amit"]
embedding_model_list=["databricks-qwen3-embedding-0-6b", "marbert-matryoshka"]
schema_list=["bertopic","default"]
input_table_list=["bertopic_input", "Arabic_offensive_comment_detection_annotation_input"]
model_result_table_list=["topic_info_local"]
instruct_model_list=["databricks-meta-llama-3-3-70b-instruct", "databricks-qwen3-next-80b-a3b-instruct"]
reduced_dimensions_list=["5","10","15"]
from pyspark.sql.functions import col
def add_column(table_name, column_name, column_type):
    try:
        if(spark.sql(f"DESCRIBE {table_name}").filter(col("col_name") == column_name).count()>0):        
            print(f"Column {column_name} already exists in table {table_name}")
        else:
            print(f"Adding column {column_name} to table {table_name}")
            spark.sql(f"""
            ALTER TABLE {table_name}
            ADD COLUMNS ({column_name} {column_type})
            """)
            print(f"Column {column_name} added")
    except Exception as e:
        print(f"Error adding column {column_name} to table {table_name}: {e}")



import time

def get_ai_merge_sql(table_name, model_name, ai_result_column, input_column, custom_expression=None):
    if custom_expression:
        expression = custom_expression
    else:
        expression = f'ai_query("{model_name}", {input_column})'
        
    # Dynamically reference the new status column
    status_column = f"{ai_result_column}_status"

    sql_str = f"""  MERGE INTO {table_name} t
    USING (
      SELECT
        id,
        {expression} AS ai_output
      FROM {table_name}
      WHERE {ai_result_column} IS NULL
        AND ({status_column} IS NULL OR {status_column} != 'FAILED')
        AND {input_column} IS NOT NULL AND {input_column} != ''
      LIMIT 300
    ) s
    ON t.id = s.id
    WHEN MATCHED AND t.{ai_result_column} IS NULL THEN
      UPDATE SET 
        t.{ai_result_column} = s.ai_output,
        -- Check the AI output: if it failed/blocked and returned NULL, mark it FAILED
        t.{status_column} = CASE WHEN s.ai_output IS NULL THEN 'FAILED' ELSE 'SUCCESS' END;
    """
    return sql_str

def add_column_batch(table_name, model_name, ai_result_column, input_column, column_type, is_ai, operation="query", target_lang="en"):
    # 1. Add your primary AI result column
    add_column(table_name, column_name=ai_result_column, column_type=column_type)
    
    # 2. Add the dedicated status tracking column as a standard STRING
    status_column = f"{ai_result_column}_status"
    add_column(table_name, column_name=status_column, column_type="STRING")
    
    custom_expr = None
    if operation == "translate":
        custom_expr = f"ai_translate({input_column}, '{target_lang}')"

    sql_str = get_ai_merge_sql(
        table_name=table_name, 
        model_name=model_name, 
        ai_result_column=ai_result_column, 
        input_column=input_column,
        custom_expression=custom_expr
    )

    batch_num = 0
    while True:
        # Check for remaining NULLs, explicitly ignoring records we already marked as FAILED
        null_count = spark.sql(f"""
            SELECT COUNT(*) AS c
            FROM {table_name}
            WHERE {ai_result_column} IS NULL
            AND ({status_column} IS NULL OR {status_column} != 'FAILED')
            AND {input_column} IS NOT NULL AND {input_column} <> ''
        """).collect()[0]["c"]
        
        batch_num += 1
        print(f"Remaining records to process: {null_count}. Processing batch {batch_num}...")
        
        if null_count == 0:
            print(f"Done. No more valid records to process for {ai_result_column}.")
            break

        start = time.perf_counter()
        spark.sql(sql_str)  
        elapsed = time.perf_counter() - start
        print(f"Batch {batch_num} latency: {elapsed:.2f} sec")

from delta.tables import DeltaTable

def create_table_with_cdf(table_name, df):
    DeltaTable.createIfNotExists(spark) \
        .tableName(table_name) \
        .addColumns(df.schema) \
        .property("delta.enableChangeDataFeed", "true") \
        .execute()
def create_table(table_name, df):
    df.write.mode("overwrite").saveAsTable(table_name)
def get_table(table_name):
    return DeltaTable.forName(spark, table_name)
def get_table_properties(table_name):
    return get_table(table_name).properties 
def write_to_table(table_name, df, write_mode="append", overwriteSchema="true"):
    create_table_with_cdf(table_name, df )
    df.write \
      .mode(write_mode) \
      .option("mergeSchema", "true") \
      .saveAsTable(table_name)

def read_checkpoint(path: str) -> int | None:
    try:
        value = dbutils.fs.head(path).strip()
        return int(value)
    except Exception:
        return None

def write_checkpoint(path: str, version: int) -> None:
    dbutils.fs.put(path, str(version), overwrite=True)
def get_current_version(table_name):
    return (
        spark.sql(f"DESCRIBE HISTORY {table_name}")
        .selectExpr("max(version) as version")
        .first()["version"]
    )

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, FloatType

# Config
batch_size = 100
truncate_dim = 256  # Optional: set to None for full 768

# Load model once

# Create target table if not exists
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {target_table} (
        id STRING,
        text_embedding ARRAY<FLOAT>,
        embedding_status STRING
    )
""")

def get_pending_count():
    """Count records not yet embedded."""
    return spark.sql(f"""
        SELECT COUNT(*) AS c
        FROM {source_table} s
        WHERE s.text IS NOT NULL AND s.text != ''
        AND NOT EXISTS (
            SELECT 1 FROM {target_table} t WHERE t.id = s.id
        )
    """).collect()[0]["c"]

def get_next_batch():
    """Get next batch — uses subquery to always read fresh target state."""
    return spark.sql(f"""
        SELECT id, text
        FROM {source_table}
        WHERE text IS NOT NULL AND text != ''
        AND id NOT IN (SELECT id FROM {target_table})
        LIMIT {batch_size}
    """).toPandas()

def get_pending_count():
    """Count remaining records."""
    return spark.sql(f"""
        SELECT COUNT(*) AS c
        FROM {source_table}
        WHERE text IS NOT NULL AND text != ''
        AND id NOT IN (SELECT id FROM {target_table})
    """).collect()[0]["c"]

def embed_and_write_batch(batch_df):
    """Embed texts and merge into target table."""
    texts = batch_df["text"].tolist()
    ids = batch_df["id"].tolist()

    try:
        embeddings = model.encode(texts, batch_size=batch_size, show_progress_bar=False)
        results = [
            {"id": doc_id, "text_embedding": emb.tolist(), "embedding_status": "SUCCESS"}
            for doc_id, emb in zip(ids, embeddings)
        ]
    except Exception as e:
        print(f"Error encoding batch: {e}")
        results = [
            {"id": doc_id, "text_embedding": None, "embedding_status": "FAILED"}
            for doc_id in ids
        ]

    schema = StructType([
        StructField("id", StringType(), False),
        StructField("text_embedding", ArrayType(FloatType()), True),
        StructField("embedding_status", StringType(), False),
    ])

    result_df = spark.createDataFrame(results, schema=schema)
    result_df.createOrReplaceTempView("batch_results")

    # MERGE — idempotent, safe to re-run
    spark.sql(f"""
        MERGE INTO {target_table} t
        USING batch_results s
        ON t.id = s.id
        WHEN NOT MATCHED THEN
            INSERT (id, text_embedding, embedding_status)
            VALUES (s.id, s.text_embedding, s.embedding_status)
    """)